# Assigning colors to annotation of pictures to dataclean

# Import libraries

In [1]:
from pathlib import Path
import json
from pathlib import Path
import json
from PIL import Image, ImageDraw

In [2]:
COLOR_LIST = [
    (220, 20, 60),    # crimson
    (65, 105, 225),   # royal_blue
    (34, 139, 34),    # forest_green
    (255, 215, 0),    # gold
    (255, 140, 0),    # dark_orange
    (147, 112, 219),  # medium_purple
    (255, 20, 147),   # deep_pink
    (64, 224, 208),   # turquoise
    (255, 99, 71),    # tomato
    (106, 90, 205),   # slate_blue
    (50, 205, 50),    # lime_green
    (218, 112, 214),  # orchid
    (70, 130, 180),   # steel_blue
    (255, 127, 80),   # coral
    (0, 139, 139),    # dark_cyan
    (244, 164, 96),   # sandy_brown
    (30, 144, 255),   # dodger_blue
    (60, 179, 113),   # medium_sea_green
    (255, 105, 180),  # hot_pink
    (218, 165, 32),   # goldenrod
]

COLOR_NAMES = [
    "crimson",
    "royal_blue",
    "forest_green",
    "gold",
    "dark_orange",
    "medium_purple",
    "deep_pink",
    "turquoise",
    "tomato",
    "slate_blue",
    "lime_green",
    "orchid",
    "steel_blue",
    "coral",
    "dark_cyan",
    "sandy_brown",
    "dodger_blue",
    "medium_sea_green",
    "hot_pink",
    "goldenrod",
]

DATASET_DIR = Path("../data/dataset")
LABELS_DIR = DATASET_DIR / "labels"
COLORED_LABELS_DIR = DATASET_DIR / "labels_colored"
COLORED_LABELS_DIR.mkdir(exist_ok=True)

label_files = sorted([p for p in LABELS_DIR.iterdir() if p.suffix == ".txt"])

for label_file in label_files:
    with open(label_file, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    colored_entries = []

    for idx, line in enumerate(lines):
        parts = line.split()
        class_id = int(parts[0])
        x_center, y_center, width, height = map(float, parts[1:])

        color = COLOR_LIST[idx % len(COLOR_LIST)]
        color_name = COLOR_NAMES[idx % len(COLOR_NAMES)]

        colored_entries.append({
            "instance_id": idx,
            "class_id": class_id,
            "class_name": "pangolin",
            "color_name": color_name,
            "color_rgb": list(color),
            "x_center": x_center,
            "y_center": y_center,
            "width": width,
            "height": height,
        })

    output_path = COLORED_LABELS_DIR / label_file.name
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(colored_entries, f, indent=2)

print(f"Saved {len(label_files)} colored label files to {COLORED_LABELS_DIR}")

Saved 1527 colored label files to ../data/dataset/labels_colored


In [3]:

# Paths
DATASET_DIR = Path("../data/dataset")
IMAGES_DIR = DATASET_DIR / "images"
COLORED_LABELS_DIR = DATASET_DIR / "labels_colored"
ANNOTATED_IMAGES_DIR = DATASET_DIR / "images_annotated"

ANNOTATED_IMAGES_DIR.mkdir(exist_ok=True)

image_files = sorted(
    [p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in [".png", ".jpg", ".jpeg"]]
)

for image_path in image_files:
    label_path = COLORED_LABELS_DIR / f"{image_path.stem}.txt"

    if not label_path.exists():
        print(f"Skipping {image_path.name}: no label file found.")
        continue

    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    img_w, img_h = image.size

    with open(label_path, "r", encoding="utf-8") as f:
        annotations = json.load(f)

    for ann in annotations:
        x_center = ann["x_center"] * img_w
        y_center = ann["y_center"] * img_h
        box_w = ann["width"] * img_w
        box_h = ann["height"] * img_h
        color = tuple(ann["color_rgb"])

        x1 = x_center - box_w / 2
        y1 = y_center - box_h / 2
        x2 = x_center + box_w / 2
        y2 = y_center + box_h / 2

        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)

    output_path = ANNOTATED_IMAGES_DIR / image_path.name
    image.save(output_path)

#print(f"Saved annotated images to: {ANNOTATED_IMAGES_DIR}")
#print(sorted([p.name for p in ANNOTATED_IMAGES_DIR.iterdir()]))